In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedGroupKFold,
    cross_validate,
)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.svm import SVC


In [ ]:
# Find the CSV whether the notebook is opened from its own folder or the repository root.
candidate_paths = [Path("Diabetes.csv"), Path("upload/Diabetes.csv")]
data_path = next((path for path in candidate_paths if path.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        "Diabetes.csv was not found. Place it beside the notebook or in the upload folder."
    )

data = pd.read_csv(data_path)
print(f"Loaded data from: {data_path.resolve()}")


In [ ]:
data.head()

In [ ]:
print("---Information---")
data.info()

In [ ]:
print("---Numerical Description---")
data.describe(include=[np.number]).T

In [ ]:
print("---Categorical Description---")
data.describe(include=[object]).T

In [ ]:
data["Gender"].unique()

In [ ]:
data["CLASS"].unique()

In [ ]:
# Clean and standardize the target and gender columns

data["CLASS"] = data["CLASS"].astype(str).str.strip().str.upper()
data["Gender"] = data["Gender"].astype(str).str.strip().str.upper()

# Display the unique values after cleaning
print("Distribution of CLASS labels:")
print(data["CLASS"].value_counts(dropna=False))

print("\nDistribution of Gender values:")
print(data["Gender"].value_counts(dropna=False))

In [ ]:
data["CLASS"].unique()

In [ ]:
data["Gender"].unique()

In [ ]:
# Select numerical columns for EDA. Identifiers are excluded because they are not measurements.
data_numeric = (
    data.select_dtypes(include=[np.number])
    .drop(columns=["ID", "No_Pation"])
    .copy()
)
data_categorical = data.select_dtypes(include=[object]).copy()


In [ ]:
data_numeric

In [ ]:
# Visualize numeric features with boxplots and identify outliers using the IQR method
cols = data_numeric.columns.tolist()

n = len(cols)
ncols = 3
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(5 * ncols, 4 * nrows))
axes = axes.flatten()

outlier_counts = {}

for i, col in enumerate(cols):
    sns.boxplot(x=data_numeric[col], ax=axes[i], color="lightblue")
    axes[i].set_title(col)
    # IQR method
    q1 = data_numeric[col].quantile(0.25)
    q3 = data_numeric[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (data_numeric[col] < lower) | (data_numeric[col] > upper)
    outlier_counts[col] = int(mask.sum())

# hide any unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

# Print number (and percentage) of outliers per feature
for col, cnt in outlier_counts.items():
    print(f"{col}: {cnt} outliers ({cnt/len(data_numeric):.2%})")

Outlier detection (DBSCAN)

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

# Numerical features used for clustering
dbscan_features = data[
    ["AGE", "Urea", "Cr", "HbA1c", "Chol",
     "TG", "HDL", "LDL", "VLDL", "BMI"]
].copy()

# Scale the data before applying DBSCAN
feature_scaler = StandardScaler()
scaled_data = feature_scaler.fit_transform(dbscan_features)

# Train the DBSCAN model
dbscan_model = DBSCAN(
    eps=3,
    min_samples=5
)

cluster_labels = dbscan_model.fit_predict(scaled_data)

# Detect observations labeled as noise (-1)
noise_points = cluster_labels == -1
detected_outliers = data.loc[noise_points]

# Display outlier information
print("DBSCAN Outlier Detection")
print("-" * 35)

print(f"Total Outliers     : {noise_points.sum()}")
print(f"Outlier Percentage : {(noise_points.sum() / len(data)) * 100:.2f}%")

print("\nIndices of Detected Outliers:")
print(detected_outliers.index.tolist())

print("\nOutlier Records:")
print(detected_outliers)

In [ ]:
# Plot the distribution of all numerical features
numeric_columns = list(data_numeric.columns)

plots_per_row = 3
total_plots = len(numeric_columns)
rows = (total_plots + plots_per_row - 1) // plots_per_row

fig, ax = plt.subplots(rows, plots_per_row, figsize=(15, 4 * rows))
ax = ax.ravel()

for index, feature in enumerate(numeric_columns):
    sns.histplot(
        data=data_numeric,
        x=feature,
        bins=25,
        kde=True,
        ax=ax[index]
    )

    ax[index].set_title(f"{feature}\nSkewness: {data_numeric[feature].skew():.2f}")
    ax[index].set_xlabel(feature)
    ax[index].set_ylabel("Frequency")

# Remove any extra empty plots
for empty_plot in ax[total_plots:]:
    fig.delaxes(empty_plot)

plt.tight_layout()
plt.show()

In [ ]:
# Bar chart showing the number of samples in each class
class_distribution = data["CLASS"].value_counts()

labels = class_distribution.index
counts = class_distribution.values

color_map = {"N": "#0B3C5D", "P": "#8B0000", "Y": "#006400"}
my_colors = [color_map[label] for label in labels]

plt.figure(figsize=(7, 5))
sns.barplot(
    x=labels,
    y=counts,
    hue=labels,
    palette=color_map,
    legend=False,
)

plt.title("Number of Samples per Diabetes Class")
plt.xlabel("Class")
plt.ylabel("Number of Patients")

for i, value in enumerate(counts):
    plt.text(i, value + 5, str(value), ha="center", fontsize=11)

plt.show()


In [ ]:
# Pie chart showing the percentage of each class
plt.figure(figsize=(6,6))

plt.pie(
    counts,
    labels=labels,
    colors=my_colors,
    autopct="%.1f%%",
    startangle=140,
    wedgeprops={"edgecolor": "white"}
)

plt.title("Distribution of Diabetes Classes")
plt.axis("equal")

plt.show()

In [ ]:
# Display the correlation matrix for numerical features

plt.figure(figsize=(11, 8))
corr = data_numeric.corr()
upper_triangle = np.triu(np.ones(corr.shape, dtype=bool))

sns.heatmap(
    corr,
    mask=upper_triangle,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    linewidths=0.4,
    square=True,
    cbar=True,
    cbar_kws={"shrink": 0.8}
)

plt.title("Correlation Heatmap of Numerical Features", fontsize=14)
plt.xticks(rotation=40, ha="right")
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Selected numerical features for visualization
selected_features = [
    ("HbA1c", "CLASS"),
    ("BMI", "CLASS"),
    ("Chol", "CLASS"),
    ("LDL", "CLASS"),
    ("Urea", "CLASS")
]

# Create folder to store the generated plots
import os
os.makedirs("figures", exist_ok=True)

# Color scheme for diabetes classes
class_colors = { "N": "#1f77b4",  "P": "#ff7f0e", "Y": "#2ca02c" }

# Generate visualizations
for feature, target in selected_features:

    if feature not in data_numeric.columns or target not in data.columns:
        continue

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    # Boxplot
    sns.boxplot(
        data=data,
        x=target,
        y=feature,
        hue=target,
        palette=class_colors,
        legend=False,
        ax=ax1
    )

    ax1.set_title(f"{feature} by {target} (Boxplot)")
    ax1.set_xlabel(target)
    ax1.set_ylabel(feature)

    # Violin plot
    sns.violinplot(
        data=data,
        x=target,
        y=feature,
        hue=target,
        palette=class_colors,
        legend=False,
        ax=ax2
    )

    ax2.set_title(f"{feature} by {target} (Violin Plot)")
    ax2.set_xlabel(target)
    ax2.set_ylabel(feature)

    plt.tight_layout()

    plt.savefig(
        f"figures/{feature.lower()}_{target.lower()}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

In [ ]:
# Calculate summary statistics for selected features by diabetes class
features = ["AGE", "BMI", "HbA1c", "Chol", "LDL", "HDL", "TG", "Urea", "Cr"]

# Keep only the features available in the dataset
available_features = data.columns.intersection(features)

# Generate summary statistics for each class
class_summary = (
    data.groupby("CLASS")[available_features].agg(["mean", "median", "std"])
)

class_summary

In [ ]:
# Scatter matrix for selected numerical features
plot_features = ["AGE", "BMI", "HbA1c", "Chol", "LDL", "TG", "Urea", "Cr"]

# Make sure the target column is categorical
data["CLASS"] = data["CLASS"].astype("category")

# Custom colors for each class
class_colors = {"N": "#1f77b4","P": "#ff7f0e", "Y": "#2ca02c"}

pair_plot = sns.pairplot(
    data=data,
    vars=plot_features,
    hue="CLASS",
    palette=class_colors,
    diag_kind="hist",
    corner=True,
    height=2.7
)

pair_plot.fig.suptitle("Scatter Matrix of Selected Numerical Features", y=1.02, fontsize=15)

plt.show()

In [ ]:
# Plot regression relationships between selected feature pairs
feature_pairs = [
    ("Chol", "LDL"),
    ("TG", "VLDL"),
    ("BMI", "HbA1c"),
    ("AGE", "HbA1c"),
    ("Urea", "Cr")
]

for feature_x, feature_y in feature_pairs:

    if feature_x not in data.columns or feature_y not in data.columns:
        continue

    plt.figure(figsize=(7, 4.5))

    sns.regplot(
        data=data,
        x=feature_x,
        y=feature_y,
        scatter_kws={
            "color": "darkgreen",
            "alpha": 0.65,
            "s": 45
        },
        line_kws={
            "color": "darkred",
            "linewidth": 2
        }
    )

    plt.title(f"{feature_x} vs {feature_y}")
    plt.xlabel(feature_x)
    plt.ylabel(feature_y)

    plt.tight_layout()
    plt.show()

In [ ]:
# Define one consistent feature set for every model.
numeric_features = [
    "AGE", "Urea", "Cr", "HbA1c", "Chol",
    "TG", "HDL", "LDL", "VLDL", "BMI",
]
categorical_features = ["Gender"]
model_features = numeric_features + categorical_features

encoder = LabelEncoder()
y = pd.Series(
    encoder.fit_transform(data["CLASS"].astype(str)),
    index=data.index,
    name="CLASS",
)
X = data[model_features].copy()

print("Target mapping:", dict(zip(encoder.classes_, encoder.transform(encoder.classes_))))


In [ ]:
# Shared preprocessing: scale measurements and one-hot encode Gender.
def build_feature_transformer():
    return ColumnTransformer(
        transformers=[
            ("scaling", StandardScaler(), numeric_features),
            (
                "encoding",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                categorical_features,
            ),
        ],
        verbose_feature_names_out=False,
    )


In [ ]:
# Keep identical feature profiles in the same partition to prevent train/test leakage.
# Seven folds produce a test set close to 15% of the dataset.
profile_groups = pd.util.hash_pandas_object(X, index=False)

holdout_splitter = StratifiedGroupKFold(
    n_splits=7,
    shuffle=True,
    random_state=42,
)
train_indices, test_indices = next(
    holdout_splitter.split(X, y, groups=profile_groups)
)

X_train = X.iloc[train_indices].copy()
X_test = X.iloc[test_indices].copy()
y_train = y.iloc[train_indices].copy()
y_test = y.iloc[test_indices].copy()
groups_train = profile_groups.iloc[train_indices].copy()

# Use group-aware cross-validation during every hyperparameter search as well.
group_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

overlap = set(profile_groups.iloc[train_indices]) & set(profile_groups.iloc[test_indices])

print("Dataset Split Summary")
print("-" * 30)
print(f"Training samples : {len(X_train)}")
print(f"Testing samples  : {len(X_test)}")
print(f"Overlapping feature profiles: {len(overlap)}")

print()
print("Class distribution in the training set:")
print(y_train.value_counts().sort_index())

print()
print("Class distribution in the testing set:")
print(y_test.value_counts().sort_index())


Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

decision_tree_model = Pipeline([
    ("feature_transformer", build_feature_transformer()),
    (
        "decision_tree",
        DecisionTreeClassifier(random_state=42, class_weight="balanced"),
    ),
])

decision_tree_model


In [ ]:
# Tune the Decision Tree with group-aware cross-validation and macro F1.
search_parameters = {
    "decision_tree__max_depth": [5, 10, None],
    "decision_tree__min_samples_split": [2, 5],
    "decision_tree__min_samples_leaf": [1, 2],
    "decision_tree__criterion": ["gini", "entropy"],
}

dt_search = GridSearchCV(
    estimator=decision_tree_model,
    param_grid=search_parameters,
    cv=group_cv,
    scoring="f1_macro",
    n_jobs=1,
)
dt_search.fit(X_train, y_train, groups=groups_train)

predictions = dt_search.predict(X_test)
print("Best Hyperparameters:", dt_search.best_params_)
print(f"Best CV Macro F1: {dt_search.best_score_:.4f}")
print(f"Test Accuracy: {accuracy_score(y_test, predictions):.4f}")
print(f"Test Balanced Accuracy: {balanced_accuracy_score(y_test, predictions):.4f}")
print(f"Test Macro F1: {f1_score(y_test, predictions, average='macro'):.4f}")
print()
print("Classification Report")
print(classification_report(y_test, predictions, target_names=encoder.classes_))

cm = confusion_matrix(y_test, predictions)
plt.figure(figsize=(7, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=encoder.classes_,
    yticklabels=encoder.classes_,
)
plt.title("Decision Tree Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.tight_layout()
plt.show()

optimal_decision_tree = dt_search.best_estimator_


In [ ]:
# Retrieve feature importance scores from the trained Decision Tree.
tree_model = optimal_decision_tree.named_steps["decision_tree"]
importance_scores = tree_model.feature_importances_
transformer = optimal_decision_tree.named_steps["feature_transformer"]
feature_names = transformer.get_feature_names_out()

importance_table = (
    pd.DataFrame({"Feature": feature_names, "Score": importance_scores})
    .sort_values(by="Score", ascending=False)
    .reset_index(drop=True)
)

print("Feature Importance Ranking")
print("-" * 35)
print(importance_table)

plt.figure(figsize=(10, 6))
plt.barh(importance_table["Feature"], importance_table["Score"], color="steelblue")
plt.title("Feature Importance Scores")
plt.xlabel("Importance Score")
plt.ylabel("Features")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(22, 11))
plot_tree(
    decision_tree=tree_model,
    feature_names=feature_names,
    class_names=encoder.classes_,
    filled=True,
    rounded=True,
    impurity=True,
    proportion=False,
    fontsize=9,
)
plt.title("Visualization of the Trained Decision Tree", fontsize=15)
plt.tight_layout()
plt.show()


Logistic Regression

In [ ]:
logistic_model = Pipeline([
    ("feature_transformer", build_feature_transformer()),
    (
        "logistic_regression",
        LogisticRegression(
            max_iter=2000,
            random_state=42,
            class_weight="balanced",
        ),
    ),
])

logistic_model


In [ ]:
search_space = {
    "logistic_regression__C": [0.01, 0.1, 1, 10, 100],
    "logistic_regression__solver": ["lbfgs"],
}

lr_search = GridSearchCV(
    estimator=logistic_model,
    param_grid=search_space,
    scoring="f1_macro",
    cv=group_cv,
    n_jobs=1,
)
lr_search.fit(X_train, y_train, groups=groups_train)

lr_predictions = lr_search.predict(X_test)
print("Optimal Hyperparameters:", lr_search.best_params_)
print(f"Best CV Macro F1: {lr_search.best_score_:.4f}")
print(f"Test Accuracy: {accuracy_score(y_test, lr_predictions):.4f}")
print(f"Test Balanced Accuracy: {balanced_accuracy_score(y_test, lr_predictions):.4f}")
print(f"Test Macro F1: {f1_score(y_test, lr_predictions, average='macro'):.4f}")
print()
print("Classification Report")
print(classification_report(y_test, lr_predictions, target_names=encoder.classes_))

conf_matrix = confusion_matrix(y_test, lr_predictions)
plt.figure(figsize=(7, 6))
sns.heatmap(
    conf_matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=encoder.classes_,
    yticklabels=encoder.classes_,
)
plt.title("Logistic Regression Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.tight_layout()
plt.show()

best_logistic_model = lr_search.best_estimator_


In [ ]:
# Multiclass Logistic Regression has one coefficient row per class.
logistic_classifier = best_logistic_model.named_steps["logistic_regression"]
transformer = best_logistic_model.named_steps["feature_transformer"]
feature_list = transformer.get_feature_names_out()
class_names = encoder.inverse_transform(logistic_classifier.classes_)

coefficient_table = pd.DataFrame(
    logistic_classifier.coef_,
    index=class_names,
    columns=feature_list,
)
coefficient_table.index.name = "Class"

print("Logistic Regression Coefficients by Class")
print(coefficient_table)

plt.figure(figsize=(12, 4))
sns.heatmap(
    coefficient_table,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
)
plt.title("Logistic Regression Coefficients by Class")
plt.xlabel("Feature")
plt.ylabel("Class")
plt.tight_layout()
plt.show()


SVM

In [ ]:
svm_model = Pipeline([
    ("feature_transformer", build_feature_transformer()),
    (
        "svm_classifier",
        SVC(
            random_state=42,
            class_weight="balanced",
            probability=True,
        ),
    ),
])

svm_model


In [ ]:
svm_parameters = {
    "svm_classifier__C": [0.1, 1, 10],
    "svm_classifier__kernel": ["linear", "rbf"],
    "svm_classifier__gamma": ["scale", "auto"],
}

svm_search = GridSearchCV(
    estimator=svm_model,
    param_grid=svm_parameters,
    scoring="f1_macro",
    cv=group_cv,
    n_jobs=1,
)
svm_search.fit(X_train, y_train, groups=groups_train)

print("Optimal SVM Parameters:", svm_search.best_params_)
print(f"Best CV Macro F1: {svm_search.best_score_:.4f}")
optimal_svm_model = svm_search.best_estimator_


In [ ]:
svm_predictions = optimal_svm_model.predict(X_test)

print("Support Vector Machine Performance")
print("-" * 40)
print(f"Accuracy: {accuracy_score(y_test, svm_predictions):.4f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, svm_predictions):.4f}")
print(f"Macro F1: {f1_score(y_test, svm_predictions, average='macro'):.4f}")
print()
print("Classification Report")
print(classification_report(y_test, svm_predictions, target_names=encoder.classes_))

conf_matrix = confusion_matrix(y_test, svm_predictions)
plt.figure(figsize=(7, 6))
sns.heatmap(
    conf_matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=encoder.classes_,
    yticklabels=encoder.classes_,
)
plt.title("SVM Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.tight_layout()
plt.show()


Random Forest

In [ ]:
random_forest_model = Pipeline([
    ("feature_transformer", build_feature_transformer()),
    (
        "random_forest",
        RandomForestClassifier(random_state=42, class_weight="balanced"),
    ),
])

random_forest_model


In [ ]:
rf_parameters = {
    "random_forest__n_estimators": [100],
    "random_forest__max_depth": [5, None],
    "random_forest__min_samples_leaf": [1, 2],
    "random_forest__criterion": ["gini", "entropy"],
    "random_forest__max_features": ["sqrt", None],
}

rf_search = GridSearchCV(
    estimator=random_forest_model,
    param_grid=rf_parameters,
    cv=group_cv,
    scoring="f1_macro",
    n_jobs=1,
)
rf_search.fit(X_train, y_train, groups=groups_train)

rf_predictions = rf_search.predict(X_test)
print("Best Random Forest Parameters:", rf_search.best_params_)
print(f"Best CV Macro F1: {rf_search.best_score_:.4f}")
print(f"Test Accuracy: {accuracy_score(y_test, rf_predictions):.4f}")
print(f"Test Balanced Accuracy: {balanced_accuracy_score(y_test, rf_predictions):.4f}")
print(f"Test Macro F1: {f1_score(y_test, rf_predictions, average='macro'):.4f}")
print()
print("Classification Report")
print(classification_report(y_test, rf_predictions, target_names=encoder.classes_))

rf_confusion = confusion_matrix(y_test, rf_predictions)
plt.figure(figsize=(7, 6))
sns.heatmap(
    rf_confusion,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=encoder.classes_,
    yticklabels=encoder.classes_,
)
plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.tight_layout()
plt.show()

best_random_forest = rf_search.best_estimator_


In [ ]:
# Gaussian Naive Bayes evaluation using the same features and group-aware folds.
naive_bayes_model = Pipeline([
    ("feature_transformer", build_feature_transformer()),
    ("gaussian_nb", GaussianNB()),
])

cv_results = cross_validate(
    estimator=naive_bayes_model,
    X=X,
    y=y,
    groups=profile_groups,
    cv=group_cv,
    scoring=["accuracy", "balanced_accuracy", "f1_macro"],
)

print("Naive Bayes Cross-Validation Results")
print("-" * 45)
print(f"Accuracy          : {cv_results['test_accuracy'].mean():.4f} ± {cv_results['test_accuracy'].std():.4f}")
print(f"Balanced Accuracy : {cv_results['test_balanced_accuracy'].mean():.4f} ± {cv_results['test_balanced_accuracy'].std():.4f}")
print(f"Macro F1          : {cv_results['test_f1_macro'].mean():.4f} ± {cv_results['test_f1_macro'].std():.4f}")


KNN

In [ ]:
# KNN uses numerical measurements; scaling is fitted inside each fold.
X_train_knn = X_train[numeric_features]
X_test_knn = X_test[numeric_features]

knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn_classifier", KNeighborsClassifier()),
])

param_grid = {
    "knn_classifier__n_neighbors": [3, 5, 7, 9, 11, 13, 15],
    "knn_classifier__weights": ["uniform", "distance"],
    "knn_classifier__metric": ["euclidean", "manhattan"],
}

grid = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=param_grid,
    cv=group_cv,
    scoring="f1_macro",
    n_jobs=1,
)
grid.fit(X_train_knn, y_train, groups=groups_train)

best_knn_model = grid.best_estimator_
knn_predictions = best_knn_model.predict(X_test_knn)

print("Best Hyperparameters:", grid.best_params_)
print(f"Best CV Macro F1: {grid.best_score_:.4f}")
print(f"Test Accuracy: {accuracy_score(y_test, knn_predictions):.4f}")
print(f"Test Balanced Accuracy: {balanced_accuracy_score(y_test, knn_predictions):.4f}")
print(f"Test Macro F1: {f1_score(y_test, knn_predictions, average='macro'):.4f}")
print()
print("Classification Report")
print(classification_report(y_test, knn_predictions, target_names=encoder.classes_))

cm = confusion_matrix(y_test, knn_predictions)
plt.figure(figsize=(7, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=encoder.classes_,
    yticklabels=encoder.classes_,
)
plt.title("K-Nearest Neighbors Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()


## Additional Models

The following models add bagging, boosting, and neural-network approaches without requiring extra packages outside scikit-learn.


In [ ]:
from sklearn.ensemble import (
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
)

additional_models = {
    "Extra Trees": Pipeline([
        ("feature_transformer", build_feature_transformer()),
        (
            "classifier",
            ExtraTreesClassifier(
                n_estimators=300,
                class_weight="balanced",
                random_state=42,
                n_jobs=1,
            ),
        ),
    ]),
    "Gradient Boosting": Pipeline([
        ("feature_transformer", build_feature_transformer()),
        (
            "classifier",
            GradientBoostingClassifier(
                n_estimators=150,
                learning_rate=0.05,
                max_depth=2,
                random_state=42,
            ),
        ),
    ]),
    "Histogram Gradient Boosting": Pipeline([
        ("feature_transformer", build_feature_transformer()),
        (
            "classifier",
            HistGradientBoostingClassifier(
                max_iter=200,
                learning_rate=0.05,
                max_leaf_nodes=15,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]),
}

additional_results = []

for model_name, model in additional_models.items():
    model.fit(X_train, y_train)
    model_predictions = model.predict(X_test)

    additional_results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, model_predictions),
        "Balanced Accuracy": balanced_accuracy_score(y_test, model_predictions),
        "Macro F1": f1_score(y_test, model_predictions, average="macro"),
    })

additional_results_table = (
    pd.DataFrame(additional_results)
    .sort_values("Macro F1", ascending=False)
    .reset_index(drop=True)
)

additional_results_table


## Complete Model Comparison

Accuracy alone can hide weak performance on the smaller classes, so the table also reports balanced accuracy, macro F1, macro ROC-AUC, and macro average precision.


In [ ]:
from sklearn.base import clone
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.preprocessing import label_binarize

# Naive Bayes was previously evaluated with cross-validation; fit a copy on the
# training set so it can be compared on the untouched holdout test set.
fitted_naive_bayes = clone(naive_bayes_model).fit(X_train, y_train)

# Each entry stores the model and the test features it expects.
comparison_models = {
    "Decision Tree": (optimal_decision_tree, X_test),
    "Logistic Regression": (best_logistic_model, X_test),
    "SVM": (optimal_svm_model, X_test),
    "Random Forest": (best_random_forest, X_test),
    "Naive Bayes": (fitted_naive_bayes, X_test),
    "KNN": (best_knn_model, X_test[numeric_features]),
    "Extra Trees": (additional_models["Extra Trees"], X_test),
    "Gradient Boosting": (additional_models["Gradient Boosting"], X_test),
    "Histogram Gradient Boosting": (
        additional_models["Histogram Gradient Boosting"],
        X_test,
    ),
}

class_ids = np.arange(len(encoder.classes_))
y_test_binary = label_binarize(y_test, classes=class_ids)
model_probabilities = {}
comparison_rows = []

for model_name, (model, model_test_data) in comparison_models.items():
    model_predictions = model.predict(model_test_data)
    probability_scores = model.predict_proba(model_test_data)
    model_probabilities[model_name] = probability_scores

    comparison_rows.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, model_predictions),
        "Balanced Accuracy": balanced_accuracy_score(y_test, model_predictions),
        "Macro F1": f1_score(y_test, model_predictions, average="macro"),
        "Macro ROC-AUC": roc_auc_score(
            y_test,
            probability_scores,
            multi_class="ovr",
            average="macro",
        ),
        "Macro Average Precision": average_precision_score(
            y_test_binary,
            probability_scores,
            average="macro",
        ),
    })

model_comparison = (
    pd.DataFrame(comparison_rows)
    .sort_values("Macro F1", ascending=False)
    .reset_index(drop=True)
)

model_comparison.round(3)


In [ ]:
# Compare the most informative imbalance-aware metrics.
plot_data = model_comparison.set_index("Model")[[
    "Balanced Accuracy",
    "Macro F1",
    "Macro ROC-AUC",
]]

ax = plot_data.plot(
    kind="barh",
    figsize=(11, 7),
    width=0.8,
)
ax.set_title("Model Performance Comparison")
ax.set_xlabel("Score")
ax.set_ylabel("Model")
ax.set_xlim(0, 1.05)
ax.legend(loc="lower right")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()


## Multiclass ROC Curves

ROC curves are calculated using one-vs-rest probabilities. The first chart compares the models using the micro-average; the second chart shows performance separately for each class.


In [ ]:
from sklearn.metrics import auc, roc_curve

plt.figure(figsize=(10, 7))

for model_name, probability_scores in model_probabilities.items():
    false_positive_rate, true_positive_rate, _ = roc_curve(
        y_test_binary.ravel(),
        probability_scores.ravel(),
    )
    micro_auc = auc(false_positive_rate, true_positive_rate)
    plt.plot(
        false_positive_rate,
        true_positive_rate,
        linewidth=2,
        label=f"{model_name} (AUC={micro_auc:.3f})",
    )

plt.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random classifier")
plt.title("Micro-Average Multiclass ROC Curves")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.xlim(0, 1)
plt.ylim(0, 1.02)
plt.legend(loc="lower right", fontsize=8)
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, len(encoder.classes_), figsize=(18, 5), sharex=True, sharey=True)

for class_index, (class_name, axis) in enumerate(zip(encoder.classes_, axes)):
    for model_name, probability_scores in model_probabilities.items():
        false_positive_rate, true_positive_rate, _ = roc_curve(
            y_test_binary[:, class_index],
            probability_scores[:, class_index],
        )
        class_auc = auc(false_positive_rate, true_positive_rate)
        axis.plot(
            false_positive_rate,
            true_positive_rate,
            linewidth=1.8,
            label=f"{model_name} ({class_auc:.3f})",
        )

    axis.plot([0, 1], [0, 1], "k--", linewidth=1)
    axis.set_title(f"Class {class_name} vs Rest")
    axis.set_xlabel("False Positive Rate")
    axis.grid(alpha=0.25)

axes[0].set_ylabel("True Positive Rate")
axes[-1].legend(loc="lower right", fontsize=7)
fig.suptitle("Class-Specific ROC Curves", fontsize=15)
plt.tight_layout()
plt.show()


## Precision–Recall Curves

Because the classes are imbalanced, precision–recall curves are an important companion to ROC curves and focus more directly on positive-class performance.


In [ ]:
from sklearn.metrics import precision_recall_curve

plt.figure(figsize=(10, 7))

for model_name, probability_scores in model_probabilities.items():
    precision_values, recall_values, _ = precision_recall_curve(
        y_test_binary.ravel(),
        probability_scores.ravel(),
    )
    average_precision = average_precision_score(
        y_test_binary,
        probability_scores,
        average="micro",
    )
    plt.plot(
        recall_values,
        precision_values,
        linewidth=2,
        label=f"{model_name} (AP={average_precision:.3f})",
    )

plt.title("Micro-Average Precision–Recall Curves")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.xlim(0, 1)
plt.ylim(0, 1.02)
plt.legend(loc="lower left", fontsize=8)
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()
